# 1 — Interaction energies (lipid–lipid + partial Coul / LJ)

Read directly from your **`.edr`** — no trajectory needed.

**Requirement:** the per-pair terms (`Coul-SR:DOPC-DOPA`, `LJ-SR:DOPC-DOPA`, …)
only exist in the `.edr` if `energygrps = DOPC DOPA` was set in the mdp of the
run that produced it. Run the cell below to list what your `.edr` actually
contains. If the pair terms are missing, they can only be generated by a
`mdrun -rerun` over a trajectory (which you'd need to obtain).

## Setup

In [ ]:
# panedr reads a GROMACS .edr straight into a DataFrame (no gmx prompt needed):
# !pip install panedr pandas numpy matplotlib
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt
EDR = "md.edr"       # <- your energy file

## List every energy term in the .edr

In [ ]:
import panedr
df = panedr.edr_to_df(EDR)
print(df.shape[0], "frames in edr")
for c in df.columns:
    print("  ", c)

## Summarise the lipid–lipid pair terms

Set `PAIRS` to the group pairs you want. `Coul-SR` = electrostatic (partial),
`LJ-SR` = van der Waals (partial), and their sum = total lipid–lipid.

In [ ]:
PAIRS = [("DOPC","DOPA"), ("DOPC","DOPC"), ("DOPA","DOPA")]

def term(a, b, kind):
    for nm in (f"{kind}:{a}-{b}", f"{kind}:{b}-{a}"):
        if nm in df.columns:
            return nm
    return None

print(f"{'pair':<14}{'Coul-SR':>12}{'LJ-SR':>12}{'total':>12}   (kJ/mol, mean +/- SEM)")
for a, b in PAIRS:
    cn, ln = term(a,b,"Coul-SR"), term(a,b,"LJ-SR")
    if cn is None and ln is None:
        print(f"{a}-{b:<9} terms not in edr (energygrps not set for this pair)")
        continue
    coul = df[cn].values if cn else np.zeros(len(df))
    lj   = df[ln].values if ln else np.zeros(len(df))
    tot  = coul + lj
    def ms(x): return f"{x.mean():.1f}+/-{x.std()/np.sqrt(len(x)):.1f}"
    print(f"{a}-{b:<9}{ms(coul):>12}{ms(lj):>12}{ms(tot):>12}")

## Fallback: use `gmx energy` instead of panedr

If you can't install panedr, run this in a terminal and pick the terms:
```bash
gmx energy -f md.edr -o inter_DOPC-DOPA.xvg   # choose Coul-SR:DOPC-DOPA  LJ-SR:DOPC-DOPA
```
then read the `.xvg`:

In [ ]:
def read_xvg(path):
    leg, rows = [], []
    for line in open(path):
        if line.startswith("@") and " legend " in line: leg.append(line.split('"')[1])
        elif not line.startswith(("#","@")): rows.append([float(x) for x in line.split()])
    return leg, np.array(rows)

# leg, d = read_xvg("inter_DOPC-DOPA.xvg")
# for i,name in enumerate(leg):
#     c = d[:,i+1]; print(f"{name:<22}{c.mean():10.2f} +/- {c.std()/np.sqrt(len(c)):.2f} kJ/mol")